In [3]:
import pandas as pd
import numpy as np
import os

# --- Configuration ---
# Update this to your actual dataset file name
FILE_PATH = "data/records.csv" 
TARGET_COL = 'loan_default' # Assuming the target variable is named 'outcome'
X_COLS = [
    "account_age_months",
    "failed_txn_count_90d",
    "credit_utilisation_ratio",
    "avg_monthly_inflow_inr_thousands",
    "repayment_score",
    "irregular_salary_flag",
    "autopay_flag",
    "monthly_spend_inr_thousands"
] # X1 to X7

os.makedirs('outputs', exist_ok=True)

# --- Load Data ---
try:
    df = pd.read_csv(FILE_PATH)
except FileNotFoundError:
    try:
        df = pd.read_excel(FILE_PATH)
    except Exception as e:
        print(f"Error loading data: {e}. Please ensure the file exists.")
        exit()

# ==========================================
# Task 2.1: Split and standardise
# ==========================================
print("--- Task 2.1 ---")
df_0 = df[df[TARGET_COL] == 0]
df_1 = df[df[TARGET_COL] == 1]

# Split 70/30 (integer arithmetic)
n0_train = (7 * len(df_0)) // 10
n1_train = (7 * len(df_1)) // 10

train_0 = df_0.iloc[:n0_train]
test_0 = df_0.iloc[n0_train:]
train_1 = df_1.iloc[:n1_train]
test_1 = df_1.iloc[n1_train:]

train_df = pd.concat([train_0, train_1]).sort_index() # Keep file order
test_df = pd.concat([test_0, test_1]).sort_index()

# Standardise
train_mean = train_df[X_COLS].mean()
train_sd = train_df[X_COLS].std()

train_std = (train_df[X_COLS] - train_mean) / train_sd
test_std = (test_df[X_COLS] - train_mean) / train_sd # Apply train stats

# Checkpoint 2.1
train_rows = len(train_df)
train_positives = int(train_df[TARGET_COL].sum())
test_rows = len(test_df)
test_positives = int(test_df[TARGET_COL].sum())
print(f"Checkpoint 2.1: train_rows = {train_rows}, train_positives = {train_positives}, test_rows = {test_rows}, test_positives = {test_positives}")
print(f"Checkpoint 2.1 loss_epoch_0: 0.6931 (Expected from uniform initial weights)")

# Prepare matrices for logistic regression
X_train = train_std.values
y_train = train_df[TARGET_COL].values
X_test = test_std.values
y_test = test_df[TARGET_COL].values







--- Task 2.1 ---
Checkpoint 2.1: train_rows = 560, train_positives = 119, test_rows = 240, test_positives = 51
Checkpoint 2.1 loss_epoch_0: 0.6931 (Expected from uniform initial weights)


In [4]:
# ==========================================
# Task 2.2: Logistic regression from scratch
# ==========================================
print("\n--- Task 2.2 ---")
def sigmoid(z):
    z = np.clip(z, -500, 500) # Prevent overflow
    return 1 / (1 + np.exp(-z))

def train_logistic_regression(X, y, eta=0.1, epochs=1000):
    n, d = X.shape
    w = np.zeros(d)
    b = 0.0
    losses = {}
    
    for epoch in range(epochs + 1):
        z = np.dot(X, w) + b
        y_hat = sigmoid(z)
        
        # Calculate loss (binary cross-entropy)
        loss = -np.mean(y * np.log(y_hat + 1e-15) + (1 - y) * np.log(1 - y_hat + 1e-15))
        losses[epoch] = loss
        
        if epoch == epochs:
            break
            
        # Gradients
        dz = y_hat - y
        dw = np.dot(X.T, dz) / n
        db = np.sum(dz) / n
        
        # Update
        w -= eta * dw
        b -= eta * db
        
    return w, b, losses

w, b, losses = train_logistic_regression(X_train, y_train, eta=0.1, epochs=1000)

print(f"Checkpoint 2.2: loss_epoch_0 = {losses[0]:.4f}, loss_epoch_1 = {losses[1]:.4f}, loss_epoch_10 = {losses[10]:.4f}, loss_epoch_100 = {losses[100]:.4f}, loss_epoch_1000 = {losses[1000]:.4f}")

# Test BCE
z_test = np.dot(X_test, w) + b
y_hat_test = sigmoid(z_test)
test_bce = -np.mean(y_test * np.log(y_hat_test + 1e-15) + (1 - y_test) * np.log(1 - y_hat_test + 1e-15))
print(f"test_bce = {test_bce:.4f}")

# Print weights for CSV
weight_names = [f"weight_X{i}" for i in range(1, 8)] + ["bias"]
weight_vals = list(w) + [b]




--- Task 2.2 ---
Checkpoint 2.2: loss_epoch_0 = 0.6931, loss_epoch_1 = 0.6751, loss_epoch_10 = 0.5685, loss_epoch_100 = 0.3952, loss_epoch_1000 = 0.3703
test_bce = 0.3824


In [5]:
# ==========================================
# Task 2.3: Learning-rate effect
# ==========================================
print("\n--- Task 2.3 ---")
etas = [0.01, 0.1, 1.0]
loss_epoch_200_dict = {}
for eta in etas:
    _, _, l_hist = train_logistic_regression(X_train, y_train, eta=eta, epochs=200)
    loss_epoch_200_dict[eta] = l_hist[200]
    print(f"loss_epoch_200_eta_{eta} = {l_hist[200]:.4f}")



--- Task 2.3 ---
loss_epoch_200_eta_0.01 = 0.5093
loss_epoch_200_eta_0.1 = 0.3775
loss_epoch_200_eta_1.0 = 0.3703


In [6]:
# ==========================================
# Task 2.4: Odds ratios
# ==========================================
print("\n--- Task 2.4 ---")
# odds ratio = e^(weight) because features are already standardized
odds_ratios = np.exp(w)
for i, or_val in enumerate(odds_ratios, 1):
    print(f"odds_ratio_X{i} = {or_val:.4f}")



--- Task 2.4 ---
odds_ratio_X1 = 0.5745
odds_ratio_X2 = 1.7233
odds_ratio_X3 = 2.4629
odds_ratio_X4 = 0.8456
odds_ratio_X5 = 0.8039
odds_ratio_X6 = 1.2597
odds_ratio_X7 = 0.9873
odds_ratio_X8 = 1.0334


In [7]:
# ==========================================
# Task 2.5: Evaluate on test rows at two thresholds
# ==========================================
print("\n--- Task 2.5 ---")
def calc_metrics(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    
    acc = (tp + tn) / len(y_true) if len(y_true) > 0 else 0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    return tp, fp, tn, fn, acc, prec, rec, f1, spec

# Threshold 0.5
pred_05 = (y_hat_test >= 0.5).astype(int)
tp_05, fp_05, tn_05, fn_05, acc_05, prec_05, rec_05, f1_05, spec_05 = calc_metrics(y_test, pred_05)

# Threshold 0.3
pred_03 = (y_hat_test >= 0.3).astype(int)
tp_03, fp_03, tn_03, fn_03, acc_03, prec_03, rec_03, f1_03, spec_03 = calc_metrics(y_test, pred_03)

# blind accuracy (predicting 0 for everyone)
blind_accuracy = np.mean(y_test == 0)
print(f"blind_accuracy = {blind_accuracy:.4f}")



--- Task 2.5 ---
blind_accuracy = 0.7875


In [8]:
# ==========================================
# Task 2.6: Threshold sweep
# ==========================================
print("\n--- Task 2.6 ---")
sweep_results = []
for t in np.arange(0.1, 1.0, 0.1):
    pred_t = (y_hat_test >= t).astype(int)
    tp, fp, tn, fn, acc, prec, rec, f1, spec = calc_metrics(y_test, pred_t)
    tpr = rec # TPR is the same as Recall
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    sweep_results.append({
        'tau': round(t, 1), 'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn, 
        'tpr': tpr, 'fpr': fpr
    })
    
pd.DataFrame(sweep_results).to_csv('outputs/part2_thresholds.csv', index=False)
print("Saved outputs/part2_thresholds.csv")



--- Task 2.6 ---
Saved outputs/part2_thresholds.csv


In [9]:
# ==========================================
# Task 2.7: K-nearest neighbours with cross-validation
# ==========================================
print("\n--- Task 2.7 ---")
def kNN_predict(X_train, y_train, X_query, K):
    preds = []
    for query in X_query:
        # Euclidean distance
        distances = np.linalg.norm(X_train - query, axis=1)
        # Stable sort by distance (handles ties by keeping file order)
        sorted_indices = np.argsort(distances, kind='stable')
        top_k_indices = sorted_indices[:K]
        # Predict positive if at least (K+1)/2 are positive
        if np.sum(y_train[top_k_indices]) >= (K + 1) / 2:
            preds.append(1)
        else:
            preds.append(0)
    return np.array(preds)

k_values = list(range(1, 22, 2))
knn_results = []

for K in k_values:
    # 5-fold cross validation
    f1_scores = []
    for fold in range(5):
        # Split by 0-based position modulo 5
        val_mask = (np.arange(len(X_train)) % 5) == fold
        tr_mask = ~val_mask
        
        X_tr_fold, y_tr_fold = X_train[tr_mask], y_train[tr_mask]
        X_val_fold, y_val_fold = X_train[val_mask], y_train[val_mask]
        
        val_preds = kNN_predict(X_tr_fold, y_tr_fold, X_val_fold, K)
        
        # F1 calculation
        tp = np.sum((y_val_fold == 1) & (val_preds == 1))
        fp = np.sum((y_val_fold == 0) & (val_preds == 1))
        fn = np.sum((y_val_fold == 1) & (val_preds == 0))
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0
        f1_scores.append(f1)
        
    mean_f1 = np.mean(f1_scores)
    knn_results.append({'K': K, 'mean_cv_f1': mean_f1})

knn_df = pd.DataFrame(knn_results)
# Choose smallest K with highest F1
best_row = knn_df.sort_values(by=['mean_cv_f1', 'K'], ascending=[False, True]).iloc[0]
best_K = int(best_row['K'])
print(f"Chosen K: {best_K} (mean_cv_f1: {best_row['mean_cv_f1']:.4f})")

# Refit with all 560 training rows and evaluate test
test_preds_knn = kNN_predict(X_train, y_train, X_test, best_K)
kn_tp, kn_fp, kn_tn, kn_fn, kn_acc, kn_prec, kn_rec, kn_f1, _ = calc_metrics(y_test, test_preds_knn)

knn_df.to_csv('outputs/part2_knn_cv.csv', index=False)
print("Saved outputs/part2_knn_cv.csv")




--- Task 2.7 ---
Chosen K: 9 (mean_cv_f1: 0.4755)
Saved outputs/part2_knn_cv.csv


In [10]:
# ==========================================
# Task 2.8: Choose the alert threshold with money
# ==========================================
print("\n--- Task 2.8 ---")
def calc_cost(preds, y_true):
    tp = np.sum((y_true == 1) & (preds == 1))
    fp = np.sum((y_true == 0) & (preds == 1))
    fn = np.sum((y_true == 1) & (preds == 0))
    tn = np.sum((y_true == 0) & (preds == 0))
    return 50 * tp + 400 * fn

# Training probabilities
z_train = np.dot(X_train, w) + b
y_hat_train = sigmoid(z_train)

costs = []
for k in range(1, 20):
    tau = k / 20
    tr_preds = (y_hat_train >= tau).astype(int)
    tr_cost = calc_cost(tr_preds, y_train)
    costs.append({'tau': tau, 'cost': tr_cost})

cost_df = pd.DataFrame(costs)
# Pick lowest cost, tie-breaker smaller tau
best_tau_row = cost_df.sort_values(by=['cost', 'tau'], ascending=[True, True]).iloc[0]
best_tau = best_tau_row['tau']
cost_tau = best_tau_row['cost']

# Test with best_tau
test_preds_tau = (y_hat_test >= best_tau).astype(int)
cost_test_at_tau = calc_cost(test_preds_tau, y_test)

# Test with 0.5
cost_test_at_0_5 = calc_cost(pred_05, y_test)

# Test flag nobody
cost_test_flag_nobody = calc_cost(np.zeros_like(y_test), y_test)

print(f"cost_tau = {cost_tau:.4f} (at tau={best_tau})")
print(f"cost_train_at_tau = {cost_tau:.4f}")
print(f"cost_test_at_tau = {cost_test_at_tau:.4f}")
print(f"cost_test_at_0.5 = {cost_test_at_0_5:.4f}")
print(f"cost_test_flag_nobody = {cost_test_flag_nobody:.4f}")

# ==========================================
# Save part2_results.txt
# ==========================================
with open('outputs/part2_results.txt', 'w') as f:
    f.write("=== Checkpoint 2.1 ===\n")
    f.write(f"train_rows = {train_rows}, train_positives = {train_positives}, test_rows = {test_rows}, test_positives = {test_positives}\n")
    
    f.write("\n=== Task 2.2 ===\n")
    f.write(f"loss_epoch_0 = {losses[0]:.4f}\nloss_epoch_1 = {losses[1]:.4f}\nloss_epoch_10 = {losses[10]:.4f}\nloss_epoch_100 = {losses[100]:.4f}\nloss_epoch_1000 = {losses[1000]:.4f}\n")
    for name, val in zip(weight_names, weight_vals):
        f.write(f"{name} = {val:.4f}\n")
    f.write(f"test_bce = {test_bce:.4f}\n")
    
    f.write("\n=== Task 2.3 ===\n")
    for eta in etas:
        f.write(f"loss_epoch_200_eta_{eta} = {loss_epoch_200_dict[eta]:.4f}\n")
        
    f.write("\n=== Task 2.4 ===\n")
    for i, or_val in enumerate(odds_ratios, 1):
        f.write(f"odds_ratio_X{i} = {or_val:.4f}\n")
        
    f.write("\n=== Task 2.5 ===\n")
    f.write(f"tau_0.5_tp = {tp_05}\ntau_0.5_fp = {fp_05}\ntau_0.5_tn = {tn_05}\ntau_0.5_fn = {fn_05}\n")
    f.write(f"tau_0.5_accuracy = {acc_05:.4f}\ntau_0.5_precision = {prec_05:.4f}\ntau_0.5_recall = {rec_05:.4f}\ntau_0.5_f1 = {f1_05:.4f}\ntau_0.5_specificity = {spec_05:.4f}\n")
    
    f.write(f"tau_0.3_tp = {tp_03}\ntau_0.3_fp = {fp_03}\ntau_0.3_tn = {tn_03}\ntau_0.3_fn = {fn_03}\n")
    f.write(f"tau_0.3_accuracy = {acc_03:.4f}\ntau_0.3_precision = {prec_03:.4f}\ntau_0.3_recall = {rec_03:.4f}\ntau_0.3_f1 = {f1_03:.4f}\ntau_0.3_specificity = {spec_03:.4f}\n")
    
    f.write(f"blind_accuracy = {blind_accuracy:.4f}\n")
    
    f.write("\n=== Task 2.7 ===\n")
    f.write(f"knn_tp = {kn_tp}\nknn_fp = {kn_fp}\nknn_tn = {kn_tn}\nknn_fn = {kn_fn}\n")
    f.write(f"knn_accuracy = {kn_acc:.4f}\nknn_precision = {kn_prec:.4f}\nknn_recall = {kn_rec:.4f}\nknn_f1 = {kn_f1:.4f}\n")
    
    f.write("\n=== Task 2.8 ===\n")
    f.write(f"cost_tau = {cost_tau:.4f}\ncost_train_at_tau = {cost_tau:.4f}\ncost_test_at_tau = {cost_test_at_tau:.4f}\ncost_test_at_0.5 = {cost_test_at_0_5:.4f}\ncost_test_flag_nobody = {cost_test_flag_nobody:.4f}\n")

print("\nSuccessfully wrote all outputs to the 'outputs/' directory.")


--- Task 2.8 ---
cost_tau = 7350.0000 (at tau=0.05)
cost_train_at_tau = 7350.0000
cost_test_at_tau = 2550.0000
cost_test_at_0.5 = 12000.0000
cost_test_flag_nobody = 20400.0000

Successfully wrote all outputs to the 'outputs/' directory.
